# main_normal — structure-first surface split (BiRefNet-invert + Metric3D normal/depth)

**Pivot from `run.py`:** no RAM++, no GroundingDINO, no Imagen, no SAM.

1. **BiRefNet** (finetuned `epoch_10.pth`) segments *objects* → **invert** → structure (wall/floor/ceiling), *visible only*.
2. **Metric3D-v2** → per-pixel **depth + normal** in one feed-forward pass.
3. Backproject structure pixels to a **point cloud**; split by:
   - **normal direction** → floor / ceiling / wall (30° cone)
   - **HDBSCAN** on `normal ⊕ W_OFFSET·plane-offset d` → individual wall instances (offset needs depth — splits *same-faced* jogged walls, grilling Q5)
   - **2D connected components** → spatially-disjoint same-plane pieces

**Known limit:** visible-only. Wall hidden behind furniture stays a hole (no amodal — that was Imagen's job in `run.py`).
Outputs go to `image2scene/<SAMPLE>/normal_out/` (non-destructive).

**Setup:** run in the `birefnet` conda env; `pip install mmengine hdbscan` (Metric3D needs mmengine — mmcv shimmed in step 2; HDBSCAN for wall clustering).

> **Verified 2026-07-28 on `user_2`:** full pipeline runs. `epoch_10.pth` structure = **0.995** (correctly leaves the empty room as structure). Metric3D normals + HDBSCAN cleanly split **left wall / right wall / far wall / floor / ceiling**. (The cancelled `epoch_10_new.pth` was unusable — 66% of the empty room mislabelled as objects; the invert path is only as good as the object model.)

In [1]:
import os, sys, math
from pathlib import Path
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

ROOT = Path.cwd()
BIREFNET_REPO = ROOT / "hq-mat" / "BiRefNet"
CKPT = BIREFNET_REPO / "ckpts" / "hypersim" / "epoch_10.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")

# ---- config / knobs (grilling Q9 = 30 deg; Q10 = offset needs depth) ----
SAMPLE          = "user_2"          # the Q5 subwall example
OBJ_THRESH      = 0.5               # BiRefNet foreground -> object
NORMAL_CONE_DEG = 30.0             # floor/ceiling cone (angle to UP_CAM)
MIN_AREA_FRAC   = 0.002            # drop wall instances smaller than this frac of image area
# --- HDBSCAN wall clustering (feature = unit normal (+) W_OFFSET * z-scored plane offset d) ---
MIN_CLUSTER_FRAC = 0.02           # HDBSCAN min_cluster_size as frac of (fit) wall pixels
W_OFFSET         = 1.0            # weight on plane-offset d: 0 = pure-normal (same-faced merge, Q5); >0 = split by depth
HDBSCAN_NFIT     = 30000          # subsample HDBSCAN is fit on, then nearest-centroid assign on all wall px
UP_CAM          = np.array([0.0, -1.0, 0.0])  # camera-up in Metric3D cam frame (y-down); floor normal ~ UP_CAM
FOCAL_FRAC      = 0.7             # fx = fy = FOCAL_FRAC * max(H,W)
print("device:", device, "| ckpt exists:", CKPT.exists())

device: cuda | ckpt exists: True


In [2]:
# ---- 1. BiRefNet (finetuned Hypersim object segmenter) ----
if str(BIREFNET_REPO) not in sys.path:
    sys.path.insert(0, str(BIREFNET_REPO))
from models.birefnet import BiRefNet

bi = BiRefNet(bb_pretrained=False)           # don't fetch swin weights; ckpt has them
sd = torch.load(CKPT, map_location="cpu")
sd = sd.get("model", sd) if isinstance(sd, dict) else sd
missing, unexpected = bi.load_state_dict(sd, strict=False)
print(f"BiRefNet loaded | missing={len(missing)} unexpected={len(unexpected)}")
bi = bi.to(device).eval()

_bi_tf = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

@torch.no_grad()
def birefnet_structure(image: Image.Image, thresh=OBJ_THRESH):
    """Full-image BiRefNet matte -> object mask -> invert = structure (bool H,W)."""
    w, h = image.size
    pred = bi(_bi_tf(image).unsqueeze(0).to(device))[-1].sigmoid().cpu()[0, 0]
    matte = F.interpolate(pred[None, None], size=(h, w), mode="bilinear",
                          align_corners=True)[0, 0].numpy()
    objects = matte >= thresh
    return objects, ~objects   # (object_mask, structure_mask)

BiRefNet loaded | missing=0 unexpected=0


In [3]:
# ---- 2. Metric3D-v2: depth + normal in one pass ----
# Metric3D's hubconf imports mmcv 1.x (unavailable on torch 2.12/cu130). It only needs
# `Config` (-> mmengine, pure-python: `pip install mmengine`) plus a couple of unused env
# helpers. Inject a lightweight `mmcv` shim BEFORE torch.hub.load so no compiled mmcv is needed.
import types as _types
from mmengine import Config as _Cfg, DictAction as _DA
_mmcv = _types.ModuleType("mmcv"); _mu = _types.ModuleType("mmcv.utils")
_mu.Config = _Cfg; _mu.DictAction = _DA
_mu.collect_env = lambda *a, **k: {}; _mu.get_git_hash = lambda *a, **k: "0" * 7
_mmcv.utils = _mu; sys.modules["mmcv"] = _mmcv; sys.modules["mmcv.utils"] = _mu

metric3d = torch.hub.load("yvanyin/metric3d", "metric3d_vit_small", pretrain=True)
metric3d = metric3d.to(device).eval()

_M3D_IN = (616, 1064)
_M3D_MEAN = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1)
_M3D_STD  = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1)

@torch.no_grad()
def metric3d_depth_normal(rgb: np.ndarray, intr):
    """rgb uint8 HxWx3 -> (depth[H,W] meters, normal[H,W,3] unit, cam frame).
    intr = [fx, fy, cx, cy]. Standard Metric3D canonical-space recipe."""
    h, w = rgb.shape[:2]
    scale = min(_M3D_IN[0] / h, _M3D_IN[1] / w)
    rgb_r = cv2.resize(rgb, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_LINEAR)
    intr_s = [intr[0] * scale, intr[1] * scale, intr[2] * scale, intr[3] * scale]
    h2, w2 = rgb_r.shape[:2]
    pad = [123.675, 116.28, 103.53]
    ph, pw = _M3D_IN[0] - h2, _M3D_IN[1] - w2
    pt, pl = ph // 2, pw // 2
    rgb_p = cv2.copyMakeBorder(rgb_r, pt, ph - pt, pl, pw - pl, cv2.BORDER_CONSTANT, value=pad)
    t = torch.from_numpy(rgb_p.transpose(2, 0, 1)).float()
    t = ((t - _M3D_MEAN) / _M3D_STD)[None].to(device)

    depth, conf, out = metric3d.inference({"input": t})

    def _unpad_resize(x, ch):
        x = x[..., pt:_M3D_IN[0] - (ph - pt), pl:_M3D_IN[1] - (pw - pl)]
        x = F.interpolate(x.reshape(1, ch, x.shape[-2], x.shape[-1]).float(),
                          (h, w), mode="bilinear", align_corners=False)
        return x[0]

    depth = _unpad_resize(depth.squeeze()[None, None] if depth.dim() <= 2 else depth, 1)[0]
    depth = depth * (intr_s[0] / 1000.0)            # canonical -> metric
    depth = depth.clamp(0, 300).cpu().numpy()

    normal = out["prediction_normal"][:, :3]        # (B,3,H,W)
    normal = _unpad_resize(normal, 3).permute(1, 2, 0).cpu().numpy()
    normal /= (np.linalg.norm(normal, axis=2, keepdims=True) + 1e-6)
    return depth, normal

Using cache found in /home/krist/.cache/torch/hub/yvanyin_metric3d_main
/home/krist/miniconda3/envs/birefnet/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/krist/miniconda3/envs/birefnet/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
xFormers not available
xFormers not available
xFormers not available
xFormers not available


In [4]:
# ---- helpers: backproject + HDBSCAN wall splitting ----
from hdbscan import HDBSCAN

def backproject(depth, intr):
    h, w = depth.shape
    fx, fy, cx, cy = intr
    u, v = np.meshgrid(np.arange(w), np.arange(h))
    X = (u - cx) * depth / fx
    Y = (v - cy) * depth / fy
    return np.stack([X, Y, depth], -1)

def classify_surface(normal, structure, cone_deg=NORMAL_CONE_DEG, up=UP_CAM):
    """Per-pixel floor/ceiling/wall inside structure, by angle(normal, up)."""
    up = up / (np.linalg.norm(up) + 1e-9)
    ang = np.degrees(np.arccos(np.clip(normal @ up, -1, 1)))
    floor   = structure & (ang < cone_deg)
    ceiling = structure & (ang > 180 - cone_deg)
    wall    = structure & ~floor & ~ceiling
    return floor, ceiling, wall

def _wall_features(normal, points, ys, xs, w_offset):
    """Feature per wall pixel = unit normal (3) (+) w_offset * z-scored plane offset d = point.n.
    d separates same-normal walls at different depth (grilling Q5); w_offset=0 -> pure-normal."""
    n = normal[ys, xs].astype(np.float32)
    p = points[ys, xs].astype(np.float32)
    d = np.sum(p * n, axis=1)
    d = (d - d.mean()) / (d.std() + 1e-6)
    return np.concatenate([n, (w_offset * d)[:, None]], axis=1).astype(np.float32)

def split_walls(wall, normal, points, min_area=0):
    """HDBSCAN-cluster wall pixels in (normal (+) offset) space -> instance masks.
    Fit HDBSCAN on a subsample, then assign EVERY wall pixel to the nearest cluster
    centroid (vectorised — approximate_predict on ~1M px is too slow). Finally split
    spatially-disjoint same-plane pieces with 2-D connected components."""
    ys, xs = np.where(wall)
    if len(ys) == 0:
        return []
    feat = _wall_features(normal, points, ys, xs, W_OFFSET)
    rng = np.random.default_rng(0)
    n_fit = min(HDBSCAN_NFIT, len(ys))
    mcs = max(20, int(MIN_CLUSTER_FRAC * n_fit))
    idx = rng.choice(len(ys), n_fit, replace=False) if len(ys) > n_fit else np.arange(len(ys))
    sub = HDBSCAN(min_cluster_size=mcs).fit(feat[idx])
    labs = sorted(set(sub.labels_) - {-1})
    if not labs:
        return []
    cents = np.stack([feat[idx][sub.labels_ == L].mean(0) for L in labs])
    lab = np.empty(len(feat), np.int32)
    for s in range(0, len(feat), 200000):
        ch = feat[s:s + 200000]
        lab[s:s + 200000] = np.argmin(((ch[:, None, :] - cents[None]) ** 2).sum(2), 1)
    inst = []
    for L in range(len(cents)):
        sel = lab == L
        m = np.zeros(wall.shape, np.uint8)
        m[ys[sel], xs[sel]] = 1
        ncc, cc = cv2.connectedComponents(m, connectivity=8)
        for c in range(1, ncc):
            comp = cc == c
            if comp.sum() >= min_area:
                inst.append(comp)
    inst.sort(key=lambda a: -a.sum())
    return inst

In [ ]:
SAMPLE          = "user_2"          # the Q5 subwall example

In [5]:
# ---- run one sample ----
sdir = ROOT / "image2scene" / SAMPLE
img_path = sdir / "original.jpg"
image = Image.open(img_path).convert("RGB")
rgb = np.array(image)
H, W = rgb.shape[:2]
f = FOCAL_FRAC * max(H, W)
intr = [f, f, W / 2.0, H / 2.0]

objects, structure = birefnet_structure(image)
depth, normal = metric3d_depth_normal(rgb, intr)
points = backproject(depth, intr)

floor, ceiling, wall = classify_surface(normal, structure)
min_area = int(MIN_AREA_FRAC * H * W)
walls = split_walls(wall, normal, points, min_area=min_area)
print(f"{SAMPLE}: structure={structure.mean():.2f}  walls={len(walls)}  "
      f"floor={floor.sum()}  ceiling={ceiling.sum()}")

user_2: structure=1.00  walls=8  floor=632485  ceiling=272738


In [6]:
# ---- save (non-destructive) ----
out = sdir / "normal_out"
out.mkdir(exist_ok=True)
def _save(mask, name):
    cv2.imwrite(str(out / name), (mask.astype(np.uint8) * 255))
_save(floor, "floor_mask.png")
_save(ceiling, "ceiling_mask.png")
_save(structure, "structure_mask.png")
for i, m in enumerate(walls):
    _save(m, f"wall_{i:02d}.png")
print("wrote", out)

wrote /home/krist/works/refine-mask/image2scene/user_2/normal_out


In [ ]:
# ---- visualize ----
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)
ov = rgb.copy()
for m in walls:
    ov[m] = (0.45 * ov[m] + 0.55 * rng.integers(60, 255, 3)).astype(np.uint8)
ov[floor]   = (0.45 * ov[floor]   + 0.55 * np.array([0, 200, 0])).astype(np.uint8)
ov[ceiling] = (0.45 * ov[ceiling] + 0.55 * np.array([0, 120, 255])).astype(np.uint8)

# BiRefNet object mask (red over input)
obj_ov = rgb.copy()
obj_ov[objects] = (0.4 * obj_ov[objects] + 0.6 * np.array([255, 0, 0])).astype(np.uint8)

fig, ax = plt.subplots(1, 4, figsize=(24, 6))
ax[0].imshow(rgb);                  ax[0].set_title("input")
ax[1].imshow((normal * 0.5 + 0.5)); ax[1].set_title("Metric3D normal")
ax[2].imshow(obj_ov);               ax[2].set_title(f"BiRefNet objects (red)  frac={objects.mean():.3f}")
ax[3].imshow(ov);                   ax[3].set_title(f"walls={len(walls)} (rand) / floor=green / ceiling=blue")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [8]:
# # ---- batch: run every sample in users_normal/ ----
# # Reuses bi (BiRefNet), metric3d, and the helpers loaded above. Per sample writes:
# #   structure_mask.png, floor_mask.png, ceiling_mask.png, wall_00..NN.png,
# #   object_mask.jpg (BiRefNet object matte), overlay.jpg (walls rand / floor green / ceiling blue).
# import time
# USERS_DIR = ROOT / "users_normal"
# samples = sorted(p for p in USERS_DIR.iterdir() if (p / "original.jpg").exists())

# def run_sample(sdir):
#     image = Image.open(sdir / "original.jpg").convert("RGB")
#     rgb = np.array(image); Hs, Ws = rgb.shape[:2]
#     fs = FOCAL_FRAC * max(Hs, Ws); intr = [fs, fs, Ws / 2.0, Hs / 2.0]
#     objects, structure = birefnet_structure(image)
#     depth, normal = metric3d_depth_normal(rgb, intr)
#     points = backproject(depth, intr)
#     floor, ceiling, wall = classify_surface(normal, structure)
#     walls = split_walls(wall, normal, points, min_area=int(MIN_AREA_FRAC * Hs * Ws))
#     sv = lambda m, n: cv2.imwrite(str(sdir / n), (m.astype(np.uint8) * 255))
#     sv(structure, "structure_mask.png"); sv(floor, "floor_mask.png"); sv(ceiling, "ceiling_mask.png")
#     for p in sdir.glob("wall_*.png"): p.unlink()
#     for i, m in enumerate(walls): sv(m, f"wall_{i:02d}.png")
#     cv2.imwrite(str(sdir / "object_mask.jpg"), (objects.astype(np.uint8) * 255))     # BiRefNet objects
#     ov = rgb.copy().astype(np.float32); rng = np.random.default_rng(1)
#     for m in walls: ov[m] = 0.45 * ov[m] + 0.55 * rng.integers(60, 255, 3)
#     ov[floor]   = 0.45 * ov[floor]   + 0.55 * np.array([0, 200, 0])
#     ov[ceiling] = 0.45 * ov[ceiling] + 0.55 * np.array([0, 120, 255])
#     cv2.imwrite(str(sdir / "overlay.jpg"), cv2.cvtColor(ov.astype(np.uint8), cv2.COLOR_RGB2BGR))
#     return len(walls), int(floor.sum()), int(ceiling.sum())

# t0 = time.time()
# for i, sdir in enumerate(samples, 1):
#     try:
#         nw, fl, ce = run_sample(sdir)
#         print(f"[{i}/{len(samples)}] {sdir.name}: walls={nw} floor={fl} ceil={ce}")
#     except Exception as e:
#         print(f"[{i}/{len(samples)}] {sdir.name}: ERROR {type(e).__name__}: {e}")
# print(f"done {len(samples)} samples in {time.time()-t0:.0f}s")

## Calibration / knobs
- **`UP_CAM`** — if floor↔ceiling swap or wall gets labeled floor, flip the sign or view `normal` panel to read the convention. Metric3D normals are camera-frame (y-down); level camera ⇒ floor normal ≈ `(0,-1,0)`.
- **`MIN_CLUSTER_FRAC`** — HDBSCAN min cluster size (frac of fit wall px). Larger ⇒ fewer, bigger walls; smaller ⇒ more fragments.
- **`W_OFFSET`** — weight of plane-offset `d` in the HDBSCAN feature. `0` = cluster on pure normal (same-faced walls merge, Q5); raise to split parallel/jogged walls by depth.
- **`NORMAL_CONE_DEG`** (30°) — floor/ceiling cone + wall-vs-wall normal gap.
- **Metric3D size** — `metric3d_vit_small` for VRAM; `_vit_large` if the 5070 allows (co-resident with BiRefNet — load sequentially / `.cpu()` BiRefNet after step 1 if OOM).
- **Amodal** — not produced. If you need wall behind furniture, morphologically inpaint each surface mask over its object holes, or re-add a fill step.